In [ ]:
!pip install -q -U langgraph langchain langchain-core langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.3 MB/s eta 0:00:00


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    api_key="GROQ_API_KEY",
    model="llama-3.1-8b-instant",
    temperature=0
)

print("Groq Connected Successfully!")

Groq Connected Successfully!


In [ ]:
from typing import TypedDict

class TeamState(TypedDict):
    task: str
    worker_result: str
    summary: str

print("State Created!")

State Created!


In [ ]:
def worker(state: TeamState):

    prompt = f"""
Solve this math problem.

Return ONLY the final numerical answer.

Problem:
{state['task']}
"""

    answer = llm.invoke(prompt).content.strip()

    return {
        "worker_result": answer
    }

print("Worker Ready!")

Worker Ready!


In [ ]:
def supervisor(state: TeamState):

    prompt = f"""
The worker solved this problem:

{state['task']}

Worker answer:

{state['worker_result']}

Write a short one-line summary.
"""

    summary = llm.invoke(prompt).content.strip()

    return {
        "summary": summary
    }

print("Supervisor Ready!")

Supervisor Ready!


In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(TeamState)

builder.add_node("worker", worker)
builder.add_node("supervisor", supervisor)

builder.add_edge(START, "worker")
builder.add_edge("worker", "supervisor")
builder.add_edge("supervisor", END)

graph = builder.compile()

print("Graph Built Successfully!")

Graph Built Successfully!


In [ ]:
result = graph.invoke(
    {
        "task": "What is 144 divided by 12, then plus 5?"
    }
)

print("Worker Result:")
print(result["worker_result"])

print()

print("Supervisor Summary:")
print(result["summary"])

Worker Result:
144 / 12 = 12
12 + 5 = 17

Supervisor Summary:
The worker solved the problem by first dividing 144 by 12 to get 12, then adding 5 to get a final answer of 17.
